## We try to answer:

If I take the numerical relativity template as my $\xi$ in time space and add to that uncorrelated gaussian noise to get a $\xi^{\prime}$, can I seperate the two components using white noise background averaging.

The Wigner function of a white noise variable shows interference structure and is not homogeneous on small scales. But by averaging multiple realizations, it becomes more and more homogeneous.

But averging $\xi^{\prime}$ it is my intuition that interference patterns will not be smoothed out, because they are not white noise - white noise interactions, but rather white noise - waveform interactions. And in the real inference case, it would be correlated noise - waveform interactions.

In [2]:
from phase_II.utils.helpers import visualize_stress, usual_plot, Stress, unpickle_me_this, fieldify
from phase_I.utils.config_jupyter_notebooks import *
from scipy.ndimage import gaussian_filter
from scipy.signal import correlate2d
%matplotlib tk

Important variables: 
		signal_strip_time, signal_strip_strain 
		signal_strip_strain_tapered
		strain
		time_domain_strip
		N


In [3]:
nrt_strain_values = np.loadtxt("../../data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("../../data/data_txt/num_rel_template_time_values.txt") - zero_time
dt = nrt_time_values[1]-nrt_time_values[0]
time_domain = ift.RGSpace(shape=(len(nrt_time_values),), distances=dt)

In [ ]:
xi = fieldify(array=nrt_strain_values, dom=time_domain)
xi_prime = fieldify(array=xi.val + 1/5*max(xi.val)*np.random.standard_normal(xi.size), dom=time_domain)

In [ ]:
plt.plot(nrt_time_values, xi_prime.val, label="NRT + white noise")
plt.plot(nrt_time_values, xi.val, label="NRT")
usual_plot()

In [ ]:
S_mat, t_dual, f_dual = Stress(xi_prime)
t_dual += min(nrt_time_values)

In [ ]:
visualize_stress(S_mat, rows=f_dual, cols=t_dual, smooth=True)

In [4]:
def generate_white_noise_stress_matrices(number_of_matrices, time_domain, supress_print=False):
    S_mat_collection = []
    for i in range(number_of_matrices):
            print(f"Calcuting white noise stress matrix, iteration {i} out of {number_of_matrices}")
            real_space_white_noise = ift.from_random(time_domain)
            stress, _, _ = Stress(real_space_white_noise, supress_print=supress_print)
            S_mat_collection.append(np.array(stress))

    print("Done, wrapping result in numpy array")
    return np.array(S_mat_collection)


In [ ]:
white_noise_ensemble = generate_white_noise_stress_matrices(number_of_matrices=10, time_domain=time_domain)

In [ ]:
out = np.empty((11, len(nrt_time_values), len(nrt_time_values)), dtype=np.complex128)

In [ ]:
white_noise_ensemble_normed = np.array(white_noise_ensemble) * 1e-3

In [ ]:
out[:10] = white_noise_ensemble_normed
out[10:] = S_mat * 1e2

In [ ]:
mean_S_mat = np.mean(out, axis=0)

In [ ]:
visualize_stress(mean_S_mat, rows=f_dual, cols=t_dual, smooth=True)

## Real case

Let's just try to do the same thing for S_mat inference, because xi_inference could be noise -> signal -> noise which is not the same as xi_prime we coded before, which is literally noise + signal all the way.

In [5]:
from phase_II.nifty_re_playground.useful.helpers import get_sample_data

S_mat_inference, t_dual_inference, f_inference = unpickle_me_this("../wigner_result_pipe_2.pickle")
time_tmp, _ = get_sample_data()
t_dual_inference = t_dual_inference + min(time_tmp)  # In my head the grav wave starts at 16.4 not at 1.4

N = len(t_dual_inference)
dt = t_dual_inference[1]-t_dual_inference[0]
time_dom = ift.RGSpace(shape=(N), distances=dt)

In [8]:
visualize_stress(S_mat_inference, rows=f_inference, cols=t_dual_inference, smooth=False)

		Rows must be in ascending order for visualization purposes but they are not, assuming a priori standard DFT order and moving DC to the middle


In [ ]:
white_noise_ensemble_for_inference = generate_white_noise_stress_matrices(number_of_matrices=10, time_domain=time_dom, supress_print=True)

In [ ]:
# Pre-allocate
out_inference = np.empty(shape=(11, N, N), dtype=np.complex128)

In [ ]:
# Fill
out_inference[:10] = white_noise_ensemble_for_inference
out_inference[10:] = S_mat_inference

In [ ]:
# Take average
S_mat_averaged_over_white_noise = np.mean(out_inference, axis=0)

In [ ]:
# S_mat_averaged_over_white_noise_smoothed = gaussian_filter(S_mat_averaged_over_white_noise, sigma=5.0)
S_mat_averaged_over_white_noise_smoothed = S_mat_averaged_over_white_noise  # no filter
np.std(S_mat_inference)

In [ ]:
# Visualize
visualize_stress(S_mat_averaged_over_white_noise_smoothed, rows=f_inference, cols=t_dual_inference, smooth=False)

In [ ]:
for Stress_matrix in white_noise_ensemble_for_inference[:2]:
    visualize_stress(Stress_matrix, rows=f_inference, cols=t_dual_inference, smooth=True)

In [ ]:
visualize_stress(np.mean(white_noise_ensemble_for_inference[:1], axis=0), rows=f_inference, cols=t_dual_inference, smooth=False)

In [ ]:
white_noise_ensemble_for_inference_smoothed = np.array([gaussian_filter(stress_matrix, sigma=5.0) for stress_matrix in white_noise_ensemble_for_inference])

In [ ]:
visualize_stress(np.mean(white_noise_ensemble_for_inference_smoothed[:10], axis=0), rows=f_inference, cols=t_dual_inference, smooth=True)

In [ ]:
literally_random_white_noise = np.random.standard_normal((N,N))
N

In [ ]:
visualize_stress(literally_random_white_noise, rows=f_inference, cols=t_dual_inference, smooth=False)

In [ ]:
# average over rows → function of x
fx = literally_random_white_noise.mean(axis=0)
print(literally_random_white_noise.shape)
# average over columns → function of y
fy = literally_random_white_noise.mean(axis=1)

covariance = np.corrcoef(fx, fy)
print(covariance)

In [ ]:
one_white_noise_wigner_realization = generate_white_noise_stress_matrices(number_of_matrices=1, time_domain=time_dom, supress_print=True)[0]

In [ ]:
# average over rows → function of x
fx = one_white_noise_wigner_realization.real.mean(axis=0)

# average over columns → function of y
fy = one_white_noise_wigner_realization.real.mean(axis=1)

covariance = np.corrcoef(fx, fy)
print(covariance)

In [ ]:
visualize_stress(one_white_noise_wigner_realization.real, rows=f_inference, cols=t_dual_inference, smooth=True)

In [ ]:
C = correlate2d(literally_random_white_noise, literally_random_white_noise, mode='full')